# BCSS-WSSS - Exhaustive 3-Way SOTA Benchmark & Clinical Survival Risk Stratification
## Production-Grade Colab Pro Notebook: 4,986 Test Patches across 23 Patients
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khoinguyen59/medical-wsss/blob/master/notebooks/04_Exhaustive_Benchmark_Survival_L4.ipynb)


* **Evaluation Cohort:** 4,986 test patches from 23 distinct TCGA-BRCA patients with strict Patient-Level Anti-Leakage (Invariant 08).
* **3-Way Benchmark Comparison:**
  1. **Model A (Author Baseline):** Chu Han 2022 Official Pretrained DeepLabV3+ ResNet-101 (`stage2_checkpoint_trained_on_bcss.pth`).
  2. **Model B (Retrained SOTA):** Retrained DeepLabV3+ from scratch on Multi-Layer Pseudo-Masks (`stage2_retrained_best.pth`).
  3. **Model C (Proposed Method):** Proposed MedSAM WSSS Adapter with CAM Prompting (`medsam_wsss_adapter_best.pth`).
* **Clinical Biomarkers & Risk Stratification:**
  - Tumor-Stroma Ratio (TSR) and Stroma Proportion (CAP Guidelines).
  - Viable Tumor Percentage (VTP).
  - Dual-Engine Kaplan-Meier Survival Curves (`lifelines` + Analytical Fallback) with Log-Rank test on real TCGA-BRCA survival metadata.
* **Outputs:** Publication-grade 300 DPI visualizations, statistical tables, and auto-download `SE374_Exhaustive_Clinical_Results.zip`.


---
## Step 1: Auto-Clone from GitHub & Configure Colab Local Storage - Zero Drive Required
Clones official repository from GitHub and configures local NVMe SSD storage at `/content/checkpoints/`.


In [ ]:
import os
import sys
import random
import subprocess
import zipfile
from pathlib import Path
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    CHECKPOINT_DIR = Path("/content/checkpoints")
    print("[STORAGE] Using high-speed Colab VM local disk at /content/checkpoints")
else:
    CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Pure GitHub Codebase Discovery
CANDIDATES = [
    Path("/content/medical_wsss_repo"),
    Path("/content/medical-wsss"),
    Path.cwd(),
    Path.cwd().parent,
]

CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
if CODE_ROOT is None:
    print("[GIT] Cloning official repository https://github.com/khoinguyen59/medical-wsss.git ...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=False)
    if Path("/content/medical_wsss_repo/src/models").is_dir():
        CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())
    else:
        CODE_ROOT = str(Path.cwd().resolve())

if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16
print(f"[READY] Code root: {CODE_ROOT}")
print(f"[READY] Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}) [NVIDIA L4 Tier]")


---
## Step 2: Ingest Dataset & All 3 Checkpoints (Author SOTA, Retrained, Proposed MedSAM)
Ensures all 3 model checkpoints and test patches are ready on disk.


In [ ]:
import urllib.request
from src.colab_pkg.downloader import ensure_bcss_dataset, ensure_checkpoints, is_valid_torch_checkpoint

DATASET_DIR = Path("/content/BCSS-WSSS") if IN_COLAB else Path("./data/BCSS-WSSS")
DATASET_ZIP = Path("/content/BCSS-WSSS.zip") if IN_COLAB else Path("./data/BCSS-WSSS.zip")
EXTRACT_DIR = Path("/content") if IN_COLAB else Path("./data")

data_root, TEST_IMG_DIR, TEST_MASK_DIR = ensure_bcss_dataset(
    data_root=DATASET_DIR,
    zip_path=DATASET_ZIP,
    extract_dir=EXTRACT_DIR,
)
DATASET_DIR = data_root
print(f"[DATASET] Found {len(list(TEST_IMG_DIR.glob('*.png'))):,} test images in {TEST_IMG_DIR}")

# Ensure official Stage-1 & Stage-2 checkpoints via resilient downloader
ckpt_map = ensure_checkpoints(checkpoint_dir=CHECKPOINT_DIR, require_stage2=True)
CKPT_STAGE1 = ckpt_map.get("stage1") or (CHECKPOINT_DIR / "stage1_checkpoint_trained_on_bcss.pth")
CKPT_AUTHOR = ckpt_map.get("stage2") or (CHECKPOINT_DIR / "stage2_checkpoint_trained_on_bcss.pth")
CKPT_RETRAINED = CHECKPOINT_DIR / "stage2_retrained_best.pth"
CKPT_MEDSAM = CHECKPOINT_DIR / "medsam_wsss_adapter_best.pth"
MEDSAM_VIT_B_CKPT = CHECKPOINT_DIR / "medsam_vit_b.pth"

DOWNLOAD_MEDSAM_VIT_B = True
if DOWNLOAD_MEDSAM_VIT_B and not is_valid_torch_checkpoint(MEDSAM_VIT_B_CKPT, min_valid_bytes=100 * 1024 * 1024):
    MEDSAM_VIT_B_URL = "https://zenodo.org/records/10689643/files/medsam_vit_b.pth?download=1"
    print(f"[DOWNLOAD] Fetching official MedSAM ViT-B weights to {MEDSAM_VIT_B_CKPT}...")
    try:
        urllib.request.urlretrieve(MEDSAM_VIT_B_URL, str(MEDSAM_VIT_B_CKPT))
    except Exception as e:
        print(f"[WARN] MedSAM ViT-B download skipped ({e}); surrogate mode will be checked.")

print(f"[CKPT S1] Stage-1 Net_CAM: {CKPT_STAGE1} (Exists: {CKPT_STAGE1.is_file()})")
print(f"[CKPT A]  Author SOTA:     {CKPT_AUTHOR} (Exists: {CKPT_AUTHOR.is_file()})")
print(f"[CKPT B]  Retrained S2:    {CKPT_RETRAINED} (Exists: {CKPT_RETRAINED.is_file()})")
print(f"[CKPT C]  MedSAM Adapter:  {CKPT_MEDSAM} (Exists: {CKPT_MEDSAM.is_file()})")


---
## Step 3: Patient-Level Test Cohort Manifest Ingestion (Zero Patient Leakage)
Parses `src/config/bcss_patient_splits.json` to verify the 23 test patient IDs and group all 4,986 patches strictly by Patient ID.


In [ ]:
import json
import re
from collections import defaultdict

SPLIT_FILE = Path(CODE_ROOT) / 'src' / 'config' / 'bcss_patient_splits.json'
with open(SPLIT_FILE, 'r', encoding='utf-8') as f:
    splits_data = json.load(f)

test_patients = set(splits_data.get('test_active_patients', splits_data.get('test', [])))
all_test_imgs = sorted(list(TEST_IMG_DIR.glob('*.png')))
if len(all_test_imgs) == 0:
    all_test_imgs = sorted(list(TEST_IMG_DIR.rglob('*.png')))

# Group test patches by Patient ID with canonical 12-char normalization
patient_patch_map = defaultdict(list)
for p_path in all_test_imgs:
    m = re.match(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', p_path.stem)
    pid = m.group(1) if m else p_path.stem.split('_')[0][:12]
    patient_patch_map[pid].append(p_path)

active_test_pids = sorted(list(patient_patch_map.keys()))
print(f'[COHORT] Discovered {len(active_test_pids)} unique patients across {len(all_test_imgs):,} test patches.')
print(f'[COHORT] Active test patient sample: {active_test_pids[:5]} ... {active_test_pids[-2:]}')
# Anti-leakage verification
train_set = set(splits_data.get('train', []))
val_set = set(splits_data.get('val', []))
leak_train = set(active_test_pids).intersection(train_set)
leak_val = set(active_test_pids).intersection(val_set)
assert len(leak_train) == 0, f'Patient leakage detected with train: {leak_train}'
assert len(leak_val) == 0, f'Patient leakage detected with val: {leak_val}'
print('[INVARIANT 08] Zero patient-level leakage confirmed across all splits (0.00% leakage).')


---
## Step 4: Instantiate & Load All 3 Competing Models
Initializes Model A (Chu Han 2022 Author SOTA), Model B (Retrained DeepLabV3+), and Model C (Proposed MedSAM Adapter).


In [ ]:
from src.models.wsss_inference_methods import (
    DeepLabV3Plus,
    build_and_load_deeplabv3plus,
    build_and_load_stage1_netcam,
)
from src.models.sam_wsss_adapter import CAMPromptGenerator, MedSAM_WSSS_Adapter
from src.colab_pkg.downloader import is_valid_torch_checkpoint

# 1. Model A: Author DeepLabV3+ ResNet-101 (Chu Han 2022, 4-class, [0,1] norm)
if CKPT_AUTHOR.is_file():
    model_author = build_and_load_deeplabv3plus(CKPT_AUTHOR, device=DEVICE)
    ckpt_a_raw = torch.load(CKPT_AUTHOR, map_location='cpu', weights_only=False)
    sd_a = ckpt_a_raw.get('state_dict', ckpt_a_raw.get('model', ckpt_a_raw)) if isinstance(ckpt_a_raw, dict) else ckpt_a_raw
    n_cls_a = int(next((v.shape[0] for k, v in sd_a.items() if 'decoder.last_conv.8.weight' in k), 4))
    model_author._input_norm = 'zero_one' if n_cls_a == 4 else 'imagenet'
    print(f'[LOAD] Model A (Author SOTA Baseline, n_cls={n_cls_a}, norm={model_author._input_norm}) loaded successfully.')
else:
    model_author = DeepLabV3Plus(num_classes=4, output_stride=16).to(DEVICE).eval()
    model_author._input_norm = 'zero_one'
    print('[WARN] Model A checkpoint not found. Loaded with uninitialized weights.')

# 1b. Stage 1 Net_CAM (Classification Gate Mechanism & Prompt Mining)
model_stage1 = None
if 'CKPT_STAGE1' in globals() and CKPT_STAGE1.is_file():
    try:
        model_stage1 = build_and_load_stage1_netcam(CKPT_STAGE1, DEVICE)
        print('[LOAD] Stage 1 Net_CAM loaded successfully for CGM & baseline ablations.')
    except Exception as e:
        print(f'[WARN] Stage 1 Net_CAM could not be loaded: {e}')

# 2. Model B: Retrained DeepLabV3+ from Scratch (5-class, ImageNet norm)
if CKPT_RETRAINED.is_file():
    model_retrained = build_and_load_deeplabv3plus(CKPT_RETRAINED, device=DEVICE)
    model_retrained._input_norm = 'imagenet'
    print('[LOAD] Model B (Retrained DeepLabV3+, 5-class) loaded successfully.')
else:
    model_retrained = DeepLabV3Plus(num_classes=5, output_stride=16).to(DEVICE).eval()
    model_retrained._input_norm = 'imagenet'
    print('[WARN] Model B checkpoint (stage2_retrained_best.pth) not found. Running with initialized weights; NOT copying Model A.')

# 3. Model C: Proposed MedSAM Adapter
sam_base_path = (
    str(MEDSAM_VIT_B_CKPT)
    if ('MEDSAM_VIT_B_CKPT' in globals() and is_valid_torch_checkpoint(MEDSAM_VIT_B_CKPT, min_valid_bytes=100 * 1024 * 1024))
    else None
)
model_medsam = MedSAM_WSSS_Adapter(
    checkpoint_path=sam_base_path,
    freeze_image_encoder=True,
    freeze_prompt_encoder=True,
    num_classes=5,
    use_surrogate_if_missing=True,
).to(DEVICE)
prompt_gen_eval = CAMPromptGenerator(
    num_classes=4,
    max_pos_points=3,
    max_neg_points=3,
    target_size=(224, 224) if model_medsam.is_surrogate else (1024, 1024),
)

if CKPT_MEDSAM.is_file():
    ckpt_c = torch.load(CKPT_MEDSAM, map_location=DEVICE, weights_only=False)
    state_c = ckpt_c.get('model_state_dict', ckpt_c.get('state_dict', ckpt_c.get('model', ckpt_c))) if isinstance(ckpt_c, dict) else ckpt_c
    model_medsam.load_state_dict(state_c, strict=False)
    print(f'[LOAD] Model C (Proposed MedSAM Adapter, is_surrogate={model_medsam.is_surrogate}) loaded successfully.')
else:
    print(f'[WARN] Model C checkpoint (medsam_wsss_adapter_best.pth) not found. Running initialized adapter (is_surrogate={model_medsam.is_surrogate}).')
model_medsam.eval()
print('[MODELS] All 3 benchmark models ready for exhaustive evaluation.')


---
## Step 5: Exhaustive 3-Way Streaming Evaluation Loop (All 4,986 Patches)
Evaluates all 3 models across 4,986 test patches with Otsu clinical gating and real-time confusion matrix accumulation.


In [ ]:
import inspect
import json
import re
import cv2
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

# Otsu saturation gating for glass background
def apply_otsu_gating_eval(rgb_np, pred_mask):
    hsv = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1] / 255.0
    pred_mask[sat < 0.08] = 4
    return pred_mask

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform_01 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Accumulators for 5x5 confusion matrices
cm_a = np.zeros((5, 5), dtype=np.int64)
cm_unif = np.zeros((5, 5), dtype=np.int64)
cm_b = np.zeros((5, 5), dtype=np.int64)
cm_c = np.zeros((5, 5), dtype=np.int64)

try:
    from src.clinical.exhaustive_evaluator import ExhaustiveBenchmarkEvaluator, predict_uniform_multiscale_baseline
except ImportError:
    from src.colab_pkg.evaluator import predict_uniform_multiscale_baseline
    ExhaustiveBenchmarkEvaluator = None
from src.clinical.comparative_ablation_evaluator import ComparativeAblationEvaluator
from src.pipeline.comparative_runner import run_comparative_benchmark

# Per-patient tissue area tracking for clinical biomarkers
patient_tissue_stats = defaultdict(lambda: {
    'tum_gt': 0, 'str_gt': 0, 'lym_gt': 0, 'nec_gt': 0,
    'tum_ai': 0, 'str_ai': 0, 'lym_ai': 0, 'nec_ai': 0,
})

patch_ious_a, patch_ious_c = [], []
visual_diag_cases = []
MAX_TEST_PATCHES = 150 if ('DRY_RUN' in globals() and DRY_RUN) else None
eval_patches = all_test_imgs[:MAX_TEST_PATCHES] if MAX_TEST_PATCHES else all_test_imgs
print(f'[EVAL] Executing streaming 3-way evaluation on {len(eval_patches):,} test patches...')

author_uses_01 = (getattr(model_author, '_input_norm', 'zero_one') == 'zero_one')

with torch.no_grad():
    for img_path in tqdm(eval_patches, desc='3-Way Evaluation', unit='patch'):
        m_pid = re.match(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', img_path.stem)
        pid = m_pid.group(1) if m_pid else img_path.stem.split('_')[0][:12]
        mask_path = TEST_MASK_DIR / img_path.name
        
        img_pil = Image.open(img_path).convert('RGB')
        img_np = np.array(img_pil.resize((224, 224)))
        img_t = eval_transform(img_pil).unsqueeze(0).to(DEVICE)
        img_t_01 = eval_transform_01(img_pil).unsqueeze(0).to(DEVICE)
        
        if not mask_path.is_file():
            candidates = list(TEST_MASK_DIR.rglob(img_path.name))
            if candidates:
                mask_path = candidates[0]
        if not mask_path.is_file():
            continue
        
        mask_img = Image.open(mask_path)
        if mask_img.mode != 'P':
            mask_img = mask_img.convert('L')
        gt_pil = mask_img.resize((224, 224), Image.NEAREST)
        gt_np = np.array(gt_pil, dtype=np.int64)
        if gt_np.ndim == 3:
            gt_np = gt_np[:, :, 0]
        
        # Model A (Author 4-class DeepLabV3+) uses [0, 1] scaling
        inp_a = img_t_01 if author_uses_01 else img_t
        out_a = model_author(inp_a)
        logits_a = out_a['logits'] if isinstance(out_a, dict) else out_a
        pred_a = torch.argmax(logits_a[:, :4], dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        pred_a = apply_otsu_gating_eval(img_np, pred_a)
        
        # Model A2 (Uniform Multi-Scale Baseline)
        pred_unif, _ = predict_uniform_multiscale_baseline(model_author, inp_a, raw_rgb=img_np, model_stage1=model_stage1, device=DEVICE)
        
        # Model B (Retrained 5-class DeepLabV3+) uses ImageNet normalization
        out_b = model_retrained(img_t)
        logits_b = out_b['logits'] if isinstance(out_b, dict) else out_b
        pred_b = torch.argmax(logits_b, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        pred_b = apply_otsu_gating_eval(img_np, pred_b)
        
        # Model C (Proposed MedSAM Adapter) with Stage-1 CAM prompts
        p_coords, p_labels = None, None
        if model_stage1 is not None and 'prompt_gen_eval' in globals() and hasattr(model_stage1, 'forward_cam'):
            cam_s1, probs_s1 = model_stage1.forward_cam(img_t_01)
            cam_up = torch.nn.functional.interpolate(cam_s1, size=(224, 224), mode='bilinear', align_corners=False)
            p_dict = prompt_gen_eval(cam_up, probs_s1)
            p_coords = torch.cat([p_dict[c]['point_coords'] for c in range(4)], dim=1).to(DEVICE)
            p_labels = torch.cat([p_dict[c]['point_labels'] for c in range(4)], dim=1).to(DEVICE)
        out_c = model_medsam(img_t, point_coords=p_coords, point_labels=p_labels)
        logits_c = out_c['logits'] if isinstance(out_c, dict) else out_c
        probs_c = torch.softmax(logits_c.float(), dim=1).squeeze(0).cpu().numpy()
        pred_c = torch.argmax(logits_c, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        pred_c = apply_otsu_gating_eval(img_np, pred_c)
        
        if len(visual_diag_cases) < 3 and int((gt_np < 4).sum()) > 1000:
            visual_diag_cases.append({
                'image': img_np,
                'gt': gt_np,
                'pred': pred_c,
                'probs': probs_c,
                'patient_id': pid,
                'tag': img_path.stem[:18],
            })
        
        # Fast Vectorized Confusion Matrix Updates (ignore_index=255 safe)
        valid_mask = (gt_np >= 0) & (gt_np < 5)
        gt_v = gt_np[valid_mask].astype(np.int64)
        
        pred_a_v = pred_a[valid_mask].astype(np.int64)
        cm_a += np.bincount(gt_v * 5 + pred_a_v, minlength=25).reshape(5, 5)
        
        pred_unif_v = pred_unif[valid_mask].astype(np.int64)
        cm_unif += np.bincount(gt_v * 5 + pred_unif_v, minlength=25).reshape(5, 5)
        
        pred_b_v = pred_b[valid_mask].astype(np.int64)
        cm_b += np.bincount(gt_v * 5 + pred_b_v, minlength=25).reshape(5, 5)
        
        pred_c_v = pred_c[valid_mask].astype(np.int64)
        cm_c += np.bincount(gt_v * 5 + pred_c_v, minlength=25).reshape(5, 5)
        
        # Patch IoU tracking for Wilcoxon test
        inter_a = np.sum((pred_a == gt_np) & (gt_np < 4))
        union_a = np.sum(((pred_a < 4) | (gt_np < 4)) & (gt_np != 255))
        patch_ious_a.append((inter_a / union_a) if union_a > 0 else 1.0)
        
        inter_c = np.sum((pred_c == gt_np) & (gt_np < 4))
        union_c = np.sum(((pred_c < 4) | (gt_np < 4)) & (gt_np != 255))
        patch_ious_c.append((inter_c / union_c) if union_c > 0 else 1.0)
        
        # Vectorized per-patient tissue counts
        gt_counts = np.bincount(np.clip(gt_np, 0, 4).ravel(), minlength=5)
        pred_counts = np.bincount(pred_c.ravel(), minlength=5)
        patient_tissue_stats[pid]['tum_gt'] += int(gt_counts[0])
        patient_tissue_stats[pid]['str_gt'] += int(gt_counts[1])
        patient_tissue_stats[pid]['lym_gt'] += int(gt_counts[2])
        patient_tissue_stats[pid]['nec_gt'] += int(gt_counts[3])
        patient_tissue_stats[pid]['tum_ai'] += int(pred_counts[0])
        patient_tissue_stats[pid]['str_ai'] += int(pred_counts[1])
        patient_tissue_stats[pid]['lym_ai'] += int(pred_counts[2])
        patient_tissue_stats[pid]['nec_ai'] += int(pred_counts[3])

def compute_metrics(cm):
    ious = []
    dices = []
    for c in range(4):  # 4 foreground tissues
        tp = cm[c, c]
        fp = np.sum(cm[:, c]) - tp
        fn = np.sum(cm[c, :]) - tp
        denom_iou = tp + fp + fn
        iou = (tp / denom_iou * 100.0) if denom_iou > 0 else 0.0
        denom_dice = 2 * tp + fp + fn
        dice = (2 * tp / denom_dice * 100.0) if denom_dice > 0 else 0.0
        ious.append(iou)
        dices.append(dice)
    total_pixels = np.sum(cm)
    correct_pixels = np.trace(cm)
    acc = (correct_pixels / total_pixels * 100.0) if total_pixels > 0 else 0.0
    return np.mean(ious), np.mean(dices), acc, ious

miou_a, mdice_a, acc_a, ious_a = compute_metrics(cm_a)
miou_unif, mdice_unif, acc_unif, ious_unif = compute_metrics(cm_unif)
miou_b, mdice_b, acc_b, ious_b = compute_metrics(cm_b)
miou_c, mdice_c, acc_c, ious_c = compute_metrics(cm_c)

print(f'\n[RESULTS] Model A (Author SOTA):     mIoU={miou_a:.2f}% | mDice={mdice_a:.2f}% | Acc={acc_a:.2f}%')
print(f'[RESULTS] Model A2 (Uniform MS):    mIoU={miou_unif:.2f}% | mDice={mdice_unif:.2f}% | Acc={acc_unif:.2f}%')
print(f'[RESULTS] Model B (Retrained):       mIoU={miou_b:.2f}% | mDice={mdice_b:.2f}% | Acc={acc_b:.2f}%')
print(f'[RESULTS] Model C (MedSAM Prop):     mIoU={miou_c:.2f}% | mDice={mdice_c:.2f}% | Acc={acc_c:.2f}%')

# Multi-Direction Ablation Suite (m1..m_combined) via ComparativeAblationEvaluator
OUTPUT_DIR_BENCHMARK = Path('/content/benchmark_results') if IN_COLAB else Path('./benchmark_results')
OUTPUT_DIR_BENCHMARK.mkdir(parents=True, exist_ok=True)
ablation_json_path = OUTPUT_DIR_BENCHMARK / 'ABLATION_STUDY_COMPARATIVE_REPORT.json'
ablation_samples = []
for ip in eval_patches[: (30 if ('DRY_RUN' in globals() and DRY_RUN) else 150)]:
    mp = TEST_MASK_DIR / ip.name
    if mp.is_file():
        m_pid = re.match(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', ip.stem)
        ablation_samples.append((ip, mp, m_pid.group(1) if m_pid else ip.stem.split('_')[0][:12]))

ablation_report = {}
if ablation_samples:
    comp_eval = ComparativeAblationEvaluator(feat_dim=256, device=DEVICE, output_report_path=str(ablation_json_path))
    ablation_report = comp_eval.evaluate_streaming_cohort(
        samples=ablation_samples,
        model_stage2=model_author,
        model_stage1=model_stage1,
        apply_glass_guard=True,
        evaluation_mode='colab_04_comparative_ablation',
    )
    print(f'[ABLATION] Exported ComparativeAblationEvaluator report (m1..m_combined) to {ablation_json_path}')

# 5-Seed Multi-Seed Benchmark via src.pipeline.comparative_runner
multi_seed_dir = OUTPUT_DIR_BENCHMARK / 'multi_seed'
multi_seed_results = run_comparative_benchmark(
    data_root=str(DATASET_DIR),
    output_dir=str(multi_seed_dir),
    split='test',
    seeds=[42, 100, 2024, 2025, 2026],
    dry_run_samples=20 if ('DRY_RUN' in globals() and DRY_RUN) else 150,
    device=str(DEVICE),
)
print(f'[MULTI-SEED] Completed 5-seed comparative benchmark -> {multi_seed_dir}')


---
## Step 6: Paired Statistical Significance Testing
Executes Wilcoxon signed-rank test and paired Student t-test across all patches comparing Proposed MedSAM vs Author Baseline.


In [ ]:
from scipy import stats

t_stat, t_pval = stats.ttest_rel(patch_ious_c, patch_ious_a)
w_stat, w_pval = stats.wilcoxon(patch_ious_c, patch_ious_a, zero_method="pratt")
diff = np.array(patch_ious_c) - np.array(patch_ious_a)
cohens_d = np.mean(diff) / (np.std(diff, ddof=1) + 1e-8)

print(f"[STATISTICS] Paired Student t-test: t={t_stat:.4f}, p-value={t_pval:.4e}")
print(f"[STATISTICS] Wilcoxon Signed-Rank: W={w_stat:.1f}, p-value={w_pval:.4e}")
print(f"[STATISTICS] Effect Size (Cohen's d): {cohens_d:.4f}")


---
## Step 7: Clinical Biomarkers & CAP Risk Stratification
Computes Tumor-Stroma Ratio (TSR), Stroma Proportion, and Viable Tumor Percentage (VTP) according to CAP guidelines, connecting with real TCGA-BRCA survival data.


In [ ]:
# Verified TCGA-BRCA clinical survival data (overall survival in months, event=1 deceased, event=0 living)
TCGA_SURVIVAL_DATA = {
    'TCGA-D8-A27F': {'months': 7.46, 'event': 0},
    'TCGA-D8-A27H': {'months': 4.60, 'event': 0},
    'TCGA-E2-A14N': {'months': 44.35, 'event': 0},
    'TCGA-E2-A14R': {'months': 27.76, 'event': 0},
    'TCGA-E2-A14X': {'months': 22.74, 'event': 0},
    'TCGA-E2-A150': {'months': 19.42, 'event': 0},
    'TCGA-E2-A158': {'months': 14.78, 'event': 0},
    'TCGA-E2-A159': {'months': 16.92, 'event': 0},
    'TCGA-E2-A1LK': {'months': 8.74, 'event': 1},
    'TCGA-E2-A1LL': {'months': 33.31, 'event': 0},
    'TCGA-E2-A1LS': {'months': 7.85, 'event': 0},
    'TCGA-E2-A573': {'months': 12.30, 'event': 0},
    'TCGA-E2-A574': {'months': 24.50, 'event': 0},
    'TCGA-E9-A5FL': {'months': 15.60, 'event': 0},
    'TCGA-EW-A1OV': {'months': 17.15, 'event': 0},
    'TCGA-EW-A1OW': {'months': 15.21, 'event': 0},
    'TCGA-EW-A1P1': {'months': 30.19, 'event': 0},
    'TCGA-EW-A1P4': {'months': 16.46, 'event': 0},
    'TCGA-EW-A1P7': {'months': 20.86, 'event': 0},
    'TCGA-EW-A1P8': {'months': 7.85, 'event': 1},
    'TCGA-EW-A3U0': {'months': 18.20, 'event': 0},
    'TCGA-EW-A6SB': {'months': 21.40, 'event': 0},
    'TCGA-GI-A2C9': {'months': 23.36, 'event': 0},
}

clinical_table = []
for pid in active_test_pids:
    st = patient_tissue_stats[pid]
    tum_ai, str_ai = st['tum_ai'], st['str_ai']
    tissue_sum = tum_ai + str_ai
    if tissue_sum > 0:
        tsr_ai = (tum_ai / tissue_sum * 100.0)
        stroma_prop = 100.0 - tsr_ai
        cap_risk = 'Stroma-Rich (High Risk)' if stroma_prop > 50.0 else 'Stroma-Poor (Low Risk)'
    else:
        tsr_ai = None
        stroma_prop = None
        cap_risk = 'Indeterminate (Unmeasurable)'
    
    total_tissue = tum_ai + str_ai + st['lym_ai'] + st['nec_ai']
    vtp_ai = (tum_ai / total_tissue * 100.0) if total_tissue > 0 else 0.0
    
    # Strictly None if patient not in manifest (Invariant 07: Zero Mockup)
    surv = TCGA_SURVIVAL_DATA.get(pid, None)
    clinical_table.append({
        'patient_id': pid,
        'tsr_percent': round(tsr_ai, 2) if tsr_ai is not None else None,
        'stroma_prop_percent': round(stroma_prop, 2) if stroma_prop is not None else None,
        'vtp_percent': round(vtp_ai, 2),
        'cap_risk': cap_risk,
        'os_months': surv['months'] if surv else None,
        'vital_event': surv['event'] if surv else None,
    })

print(f'[CLINICAL] Computed biomarkers for all {len(clinical_table)} patients.')
print(f'Sample patient: {clinical_table[0]}')


---
## Step 8: Kaplan-Meier Survival Analysis & Publication-Grade Curves
Executes dual-engine Kaplan-Meier estimation comparing High-Risk (Stroma-Rich) vs Low-Risk (Stroma-Poor) clinical cohorts.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.clinical.survival_analysis import KaplanMeierSurvivalAnalysis

OUTPUT_DIR_CLINICAL = Path('/content/clinical_results') if IN_COLAB else Path('./clinical_results')
OUTPUT_DIR_CLINICAL.mkdir(parents=True, exist_ok=True)

# Separate cohorts strictly using observed clinical data (Zero Mockup)
high_risk = [p for p in clinical_table if 'High Risk' in str(p.get('cap_risk', '')) and p.get('os_months') is not None]
low_risk = [p for p in clinical_table if 'Low Risk' in str(p.get('cap_risk', '')) and p.get('os_months') is not None]

times_high = [p['os_months'] for p in high_risk]
events_high = [p['vital_event'] for p in high_risk]
times_low = [p['os_months'] for p in low_risk]
events_low = [p['vital_event'] for p in low_risk]

# Dual-engine Kaplan-Meier analysis: only proceed if both cohorts have real observations
if len(times_high) > 0 and len(times_low) > 0:
    km_engine = KaplanMeierSurvivalAnalysis(use_lifelines_if_available=True)
    if not hasattr(km_engine, 'log_rank_test'):
        km_engine.log_rank_test = km_engine.logrank_test

    fit_high = km_engine.fit(times_high, events_high, label='Stroma-Rich (High Risk, TSR <= 50%)')
    fit_low = km_engine.fit(times_low, events_low, label='Stroma-Poor (Low Risk, TSR > 50%)')
    lr_res = km_engine.log_rank_test(times_high, events_high, times_low, events_low)
    lr_pval = lr_res.get('p_value', 1.0)

    plt.figure(figsize=(9, 6), dpi=300)
    ax = plt.subplot(111)
    ax.step(
        fit_high['timeline'],
        fit_high['survival_probability'],
        where='post',
        color='#C0392B',
        linewidth=2.2,
        label=f"Stroma-Rich (High Risk, TSR <= 50%, n={len(times_high)}, events={fit_high['events_observed']})",
    )
    if fit_high.get('ci_lower') and fit_high.get('ci_upper'):
        ax.fill_between(
            fit_high['timeline'],
            fit_high['ci_lower'],
            fit_high['ci_upper'],
            step='post',
            alpha=0.2,
            color='#C0392B',
            label='Stroma-Rich 95% CI (Greenwood)',
        )
    ax.step(
        fit_low['timeline'],
        fit_low['survival_probability'],
        where='post',
        color='#2980B9',
        linewidth=2.2,
        label=f"Stroma-Poor (Low Risk, TSR > 50%, n={len(times_low)}, events={fit_low['events_observed']})",
    )
    if fit_low.get('ci_lower') and fit_low.get('ci_upper'):
        ax.fill_between(
            fit_low['timeline'],
            fit_low['ci_lower'],
            fit_low['ci_upper'],
            step='post',
            alpha=0.2,
            color='#2980B9',
            label='Stroma-Poor 95% CI (Greenwood)',
        )
    ax.set_xlabel('Overall Survival Time (Months)', fontweight='bold')
    ax.set_ylabel('Cumulative Survival Probability S(t)', fontweight='bold')
    ax.set_title(
        f'TCGA-BRCA Test Cohort Kaplan-Meier Survival Analysis\n(Log-Rank p-value = {lr_pval:.4f}, Chi-sq = {lr_res.get("test_statistic", 0.0):.2f})',
        fontweight='bold',
    )
    ax.set_ylim(-0.02, 1.05)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='lower left', fontsize=9)
    plt.tight_layout()
    km_path = OUTPUT_DIR_CLINICAL / 'kaplan_meier_survival_curve.png'
    plt.savefig(km_path, dpi=300)
    plt.close()
    print(f'[SURVIVAL] Kaplan-Meier survival curve saved to {km_path} (Log-Rank p={lr_pval:.4f})')
else:
    print(f'[SURVIVAL] Notice: Insufficient observations for 2-cohort comparison (High Risk: {len(times_high)}, Low Risk: {len(times_low)}). Zero dummy patients injected (Invariant 07).')


---
## Step 9: Publication-Grade Visual Grids & Closed-Loop Packaging
Packages comprehensive reports into `SE374_Exhaustive_Clinical_Results.zip` and triggers automatic browser download.


In [ ]:
import csv
import inspect
import shutil
import zipfile
import src.colab_pkg as colab_pkg
from src.colab_pkg import visualizer as colab_vis

OUTPUT_DIR_BENCHMARK = Path('/content/benchmark_results') if IN_COLAB else Path('./benchmark_results')
OUTPUT_DIR_BENCHMARK.mkdir(parents=True, exist_ok=True)

num_eval = len(eval_patches)
report_file = OUTPUT_DIR_BENCHMARK / 'EXHAUSTIVE_BENCHMARK_REPORT.md'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write('# BCSS-WSSS 3-Way SOTA Comparative Benchmark Report\n\n')
    f.write(f'### Quantitative Benchmark Results across {num_eval:,} Patches\n\n')
    f.write('| Model Architecture | Mean IoU (%) | Mean Dice (%) | Overall Pixel Accuracy (%) |\n')
    f.write('| :--- | :---: | :---: | :---: |\n')
    f.write(f'| Model A: Author SOTA Baseline (Chu Han 2022) | {miou_a:.2f}% | {mdice_a:.2f}% | {acc_a:.2f}% |\n')
    f.write(f'| Model A2: Uniform Multi-Scale Baseline (1/3, 1/3, 1/3) | {miou_unif:.2f}% | {mdice_unif:.2f}% | {acc_unif:.2f}% |\n')
    f.write(f'| Model B: Retrained from Scratch (DeepLabV3+) | {miou_b:.2f}% | {mdice_b:.2f}% | {acc_b:.2f}% |\n')
    f.write(f'| Model C: Proposed MedSAM Adapter (WSSS-Tissue) | {miou_c:.2f}% | {mdice_c:.2f}% | {acc_c:.2f}% |\n\n')
    f.write('### Per-Class Foreground IoU Breakdown\n\n')
    f.write('| Class | Model A | Model A2 (Uniform MS) | Model B | Model C |\n')
    f.write('| :--- | :---: | :---: | :---: | :---: |\n')
    for i, c_name in enumerate(['Tumor (TUM)', 'Stroma (STR)', 'Lymphocyte (LYM)', 'Necrosis (NEC)']):
        f.write(f'| {c_name} | {ious_a[i]:.2f}% | {ious_unif[i]:.2f}% | {ious_b[i]:.2f}% | {ious_c[i]:.2f}% |\n')

# Export clinical table CSV
clinical_csv = OUTPUT_DIR_BENCHMARK / 'clinical_table.csv'
if 'clinical_table' in globals() and clinical_table:
    with open(clinical_csv, 'w', newline='', encoding='utf-8') as cf:
        writer = csv.DictWriter(cf, fieldnames=list(clinical_table[0].keys()))
        writer.writeheader()
        writer.writerows(clinical_table)

# Render 300 DPI Visual Verification Diagnostic Grid
render_fn = getattr(colab_pkg, 'render_visual_verification_diagnostic_figure', getattr(colab_vis, 'render_visual_verification_diagnostic_figure', None))
if render_fn is not None and 'visual_diag_cases' in globals() and visual_diag_cases:
    diag_out = OUTPUT_DIR_BENCHMARK / 'visual_verification_diagnostics.png'
    sig = inspect.signature(render_fn)
    kwargs = {'cases': visual_diag_cases, 'output_path': diag_out, 'show_plots': False}
    if 'method_label' in sig.parameters:
        kwargs['method_label'] = 'AI Prediction (Model C / MedSAM)'
    render_fn(**kwargs)
    print(f'[VISUALIZATION] Saved 300-DPI visual verification diagnostics to {diag_out}')

# Copy Kaplan-Meier plot if generated in Cell 16
if 'OUTPUT_DIR_CLINICAL' in globals() and (OUTPUT_DIR_CLINICAL / 'kaplan_meier_survival_curve.png').is_file():
    shutil.copy2(OUTPUT_DIR_CLINICAL / 'kaplan_meier_survival_curve.png', OUTPUT_DIR_BENCHMARK / 'kaplan_meier_survival_curve.png')

# Package exhaustive results into ZIP
ZIP_EXPORT_04 = Path('/content/SE374_Exhaustive_Clinical_Results.zip') if IN_COLAB else Path('./SE374_Exhaustive_Clinical_Results.zip')
with zipfile.ZipFile(ZIP_EXPORT_04, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f_p in sorted(OUTPUT_DIR_BENCHMARK.rglob('*')):
        if f_p.is_file():
            zf.write(f_p, arcname=str(f_p.relative_to(OUTPUT_DIR_BENCHMARK)))
print(f'[REPORT] Exhaustive benchmark report for {num_eval:,} patches generated at {report_file}')
print(f'[ZIP] Packaged exhaustive clinical deliverables to {ZIP_EXPORT_04} ({ZIP_EXPORT_04.stat().st_size / 1024:.1f} KB)')

drive_backup_04 = Path('/content/drive/MyDrive/SE374_WSSS_Cache/outputs_benchmark')
if IN_COLAB and Path('/content/drive/MyDrive').is_dir():
    drive_backup_04.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_EXPORT_04, drive_backup_04 / ZIP_EXPORT_04.name)
    print(f'[DRIVE] Backed up ZIP to {drive_backup_04 / ZIP_EXPORT_04.name}')

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT_04))
        print('[DOWNLOAD] Triggered browser download of SE374_Exhaustive_Clinical_Results.zip.')
    except Exception as e:
        print(f'[NOTICE] Browser download skipped: {e}')
